In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu")

print(f"Using {device} device")

Using mps device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [6]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([1], device='mps:0')


In [7]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


In [8]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size()
      )

torch.Size([3, 784])


In [9]:
layer1 = nn.Linear(in_features=28*28, out_features = 20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [10]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1361, -0.2788, -0.1386,  0.4284, -0.6422, -0.2361,  0.1173, -0.0687,
          0.2106, -0.8613, -0.0171,  0.3026, -0.5611,  0.0699,  0.3520,  0.5511,
          0.0314,  0.4699,  0.0488, -0.2726],
        [-0.0505,  0.1295,  0.0156,  0.2880, -0.5965, -0.1333, -0.1016, -0.1140,
         -0.0109, -1.0010,  0.0295,  0.1728, -0.3716,  0.0591,  0.4963,  0.4349,
          0.3216,  0.3789, -0.1004, -0.2506],
        [-0.0337, -0.1989,  0.1459, -0.0660, -0.3505,  0.0617,  0.0175, -0.1433,
          0.2607, -0.6406, -0.2103, -0.2426, -0.5338,  0.0110,  0.6032,  0.2608,
         -0.1320,  0.2771, -0.0659, -0.3169]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.0000, 0.0000, 0.4284, 0.0000, 0.0000, 0.1173, 0.0000, 0.2106,
         0.0000, 0.0000, 0.3026, 0.0000, 0.0699, 0.3520, 0.5511, 0.0314, 0.4699,
         0.0488, 0.0000],
        [0.0000, 0.1295, 0.0156, 0.2880, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0295, 0.1728, 0.0000, 0.0591, 0.49

In [11]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [12]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)


In [13]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0242, -0.0148,  0.0336,  ...,  0.0129, -0.0017, -0.0186],
        [-0.0004, -0.0104,  0.0342,  ..., -0.0099,  0.0073,  0.0068]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0090, -0.0227], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0243, -0.0173, -0.0255,  ...,  0.0241,  0.0036,  0.0054],
        [-0.0109, -0.0089,  0.0186,  ...,  0.0366,  0.0191, -0.0365]],
       device='mps:0', grad_fn=<Slice